# Sprint 2–4: Simulation

Generate virtual users → run goal-driven agents → persist interactions to SQLite.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import yaml
from dotenv import load_dotenv
load_dotenv('../.env')  # HF_TOKEN must be set

with open('../configs/mind_small.yaml') as f:
    config = yaml.safe_load(f)
print('HF_TOKEN set:', bool(os.getenv('HF_TOKEN')))

In [ ]:
# Generate virtual user profiles
from simulation.user_generator import generate_users

N_USERS = 10  # Start small for testing; set to config['dataset']['max_virtual_users'] for full run
virtual_users = generate_users(n=N_USERS, seed=config['simulation']['seed'])
print(f'Generated {len(virtual_users)} virtual users')
for u in virtual_users[:3]:
    print(f"  {u['user_id']}: [{u['role']}] ({u['language_pref']}) — {u['goal'][:60]}...")

In [ ]:
# Quick smoke-test: run 1 user × 1 session manually before full run
import pickle
import numpy as np
import sqlite3
from sentence_transformers import SentenceTransformer
from data.news_preprocessor import load_index
from recsim_env.document import MindDocumentSampler
from recsim_env.user_sampler import MindUserSampler
from recsim_env.environment import build_environment
from simulation.agent import GoalDrivenAgent
from simulation.db import init_db, insert_user

ds = config['dataset']
ir = config['ir']
sim_cfg = config['simulation']

index, corpus_ids, embeddings, db_con = load_index(
    ds['index_path'], ds['ids_path'], ds['embeddings_path'], ds['db_path']
)
embed_model = SentenceTransformer(ir['model'])
print(f'Index loaded: {index.ntotal:,} docs')

In [ ]:
# Single-user smoke-test (1 session)
test_user = virtual_users[0]
print(f"Testing with: {test_user['user_id']} | {test_user['role']} | {test_user['language_pref']}")
print(f"Goal: {test_user['goal']}")
print(f"Starting query: {test_user['starting_query']}")

doc_sampler = MindDocumentSampler(
    faiss_index=index, corpus_ids=corpus_ids, embeddings=embeddings,
    db_con=db_con, embed_model=embed_model,
    num_candidates=sim_cfg['num_candidates'], seed=sim_cfg['seed']
)
user_sampler = MindUserSampler([test_user], embed_model, seed=sim_cfg['seed'])
env = build_environment(user_sampler, doc_sampler, sim_cfg['slate_size'],
                         sim_cfg['num_candidates'], sim_cfg['seed'])
agent = GoalDrivenAgent(test_user, slate_size=sim_cfg['slate_size'], max_steps=3)

doc_sampler.set_query(test_user['starting_query'])
obs = env.reset()
print('\nRecSim env reset OK.')
print('User obs keys:', list(obs.get('user', {}).keys()))
print('Doc obs count:', len(obs.get('doc', {})))

In [ ]:
# Run 3 steps
for step in range(1, 4):
    slate = agent.select_slate(obs)
    next_q = agent.get_next_query()
    doc_sampler.set_query(next_q)
    obs, reward, done, info = env.step(slate)
    responses = info.get('response', [])
    clicks = sum(1 for r in responses if isinstance(r, dict) and r.get('clicked', 0))
    print(f'  step {step}: slate={list(slate)}, reward={reward:.3f}, clicks={clicks}, next_q="{next_q[:50]}"')
    if done or agent.should_stop:
        print('  Session ended.')
        break
print('Smoke-test passed.')

In [ ]:
# Full simulation run
from simulation.runner import run_simulation

results = run_simulation(
    config=config,
    virtual_users=virtual_users,
    parallel_workers=1,  # set to sim_cfg['parallel_workers'] for full run
)
ok = sum(1 for r in results if r.get('ok'))
print(f'{ok}/{len(results)} users completed successfully.')

In [ ]:
# Inspect persisted interactions
from simulation.db import fetch_all_interactions
import pandas as pd

interactions = fetch_all_interactions(sim_cfg['sim_db_path'])
df = pd.DataFrame(interactions)
print(f'Total interactions: {len(df):,}')
print(f'Click rate: {df["clicked"].mean():.2%}')
print(f'Mean relevance (clicked): {df[df["clicked"]==1]["relevance"].mean():.3f}')
print(df[['user_id','session_num','step','clicked','relevance','fatigue']].head(10))